# Facebook Comment Bot - Step by Step Testing
Run the cells below one by one to test the bot functionality.

In [1]:
import os
import random
import time
import logging
from datetime import datetime

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from webdriver_manager.chrome import ChromeDriverManager
import openai
from dotenv import load_dotenv

load_dotenv()
print('Imports successful.')

Imports successful.


In [2]:
CONFIG = {
    'POST_URL': os.getenv('POST_URL'),
    'COMMENT_BOX_XPATH': "//div[contains(@aria-label, 'Write a comment') and @contenteditable='true']",
    'MAX_COMMENTS': 100,
    'MAX_ITERATIONS': 10000,
    'DELAYS': {
        'SHORT_MIN': 0.5,
        'SHORT_MAX': 2.0,
        'MEDIUM_MIN': 1,
        'MEDIUM_MAX': 3,
        'LONG_MIN': 5,
        'LONG_MAX': 20,
        'RELOAD_PAUSE': 180,
    },
    'CHROME_PROFILE': 'Default'
}

OPENAI_CONFIG = {
    'API_KEY': os.getenv('OPENAI_API_KEY'),
    'MODEL': os.getenv('OPENAI_MODEL'),
    'PROMPT': os.getenv('OPENAI_PROMPT') + 'Do not include emojis or any introductory phrases or additional text.'
}

print('Config loaded.')

TypeError: unsupported operand type(s) for +: 'NoneType' and 'str'

In [ ]:
def setup_logger():
    os.makedirs('logs', exist_ok=True)
    log_filename = f'logs/facebook_comment_bot_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log'
    logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s: %(message)s', handlers=[logging.FileHandler(log_filename, encoding='utf-8'), logging.StreamHandler()])
    return logging.getLogger(__name__)

logger = setup_logger()

In [ ]:
class FacebookAICommentBot:
    def __init__(self, config=None):
        self.config = {**CONFIG, **(config or {})}
        self.driver = None
        openai.api_key = OPENAI_CONFIG['API_KEY']

    def setup_driver(self):
        chrome_options = Options()
        chrome_options.add_argument('--disable-popup-blocking')
        chrome_options.add_argument('--disable-notifications')
        chrome_options.add_experimental_option('excludeSwitches', ['enable-automation'])
        chrome_options.add_experimental_option('useAutomationExtension', False)
        chrome_options.binary_location = 'C:/Program Files/Google/Chrome/Application/chrome.exe'
        user_data_dir = os.path.join(os.getcwd(), 'chrome_data')
        chrome_options.add_argument(f'--user-data-dir={user_data_dir}')
        chrome_options.add_argument(f'--profile-directory={self.config["CHROME_PROFILE"]}')
        service = Service(ChromeDriverManager().install())
        self.driver = webdriver.Chrome(service=service, options=chrome_options)
        logger.info('Chrome driver set up successfully.')

    def random_pause(self, min_time=1, max_time=5):
        delay = random.uniform(min_time, max_time)
        time.sleep(delay)
        logger.debug(f'Paused for {delay:.2f} seconds.')

    def human_mouse_jiggle(self, element, moves=2):
        actions = ActionChains(self.driver)
        actions.move_to_element(element).perform()
        for _ in range(moves):
            x_offset = random.randint(-15, 15)
            y_offset = random.randint(-15, 15)
            actions.move_by_offset(x_offset, y_offset).perform()
            self.random_pause(0.3, 1)
        actions.move_to_element(element).perform()
        self.random_pause(0.3, 1)

    def human_type(self, element, text):
        words = text.split()
        for w_i, word in enumerate(words):
            if random.random() < 0.05:
                fake_word = random.choice(['aaa', 'zzz', 'hmm'])
                for c in fake_word:
                    element.send_keys(c)
                    time.sleep(random.uniform(0.08, 0.35))
                for _ in fake_word:
                    element.send_keys(Keys.BACKSPACE)
                    time.sleep(random.uniform(0.06, 0.25))
            for char in word:
                if random.random() < 0.05:
                    wrong_char = random.choice('abcdefghijklmnopqrstuvwxyz')
                    element.send_keys(wrong_char)
                    time.sleep(random.uniform(0.08, 0.35))
                    element.send_keys(Keys.BACKSPACE)
                    time.sleep(random.uniform(0.06, 0.25))
                element.send_keys(char)
                time.sleep(random.uniform(0.08, 0.35))
            if w_i < len(words) - 1:
                element.send_keys(' ')
                time.sleep(random.uniform(0.08, 0.3))
            if random.random() < 0.03:
                element.send_keys(Keys.ARROW_LEFT)
                time.sleep(random.uniform(0.1, 0.3))
                element.send_keys(Keys.ARROW_RIGHT)
                time.sleep(random.uniform(0.1, 0.3))
        self.random_pause(0.5, 1.5)

    def random_scroll(self):
        scroll_distance = random.randint(200, 800)
        if random.choice(['up', 'down']) == 'down':
            self.driver.execute_script(f'window.scrollBy(0, {scroll_distance});')
        else:
            self.driver.execute_script(f'window.scrollBy(0, -{scroll_distance});')
        self.random_pause(1, 3)

    def random_hover_or_click(self):
        all_links = self.driver.find_elements(By.TAG_NAME, 'a')
        if all_links and random.random() < 0.5:
            random_link = random.choice(all_links)
            try:
                ActionChains(self.driver).move_to_element(random_link).perform()
                self.random_pause(1, 3)
                if random.random() < 0.2:
                    random_link.click()
                    time.sleep(3)
                    self.driver.back()
                    self.random_pause(1, 3)
            except Exception: pass

    def generate_comment(self):
        try:
            response = openai.ChatCompletion.create(model=OPENAI_CONFIG['MODEL'], messages=[{'role': 'user', 'content': OPENAI_CONFIG['PROMPT']}])
            return response.choices[0].message['content'].strip()
        except Exception:
            return 'Such a thoughtful post! Thanks for sharing! 😊'

    def post_comment(self, comment, comment_count):
        try:
            comment_area = WebDriverWait(self.driver, 10).until(EC.presence_of_element_located((By.XPATH, self.config['COMMENT_BOX_XPATH'])))
            if random.random() < 0.4:
                self.random_scroll()
            else:
                self.random_hover_or_click()
            self.human_mouse_jiggle(comment_area, moves=3)
            comment_area.click()
            self.random_pause(0.5, 2.0)
            self.human_type(comment_area, comment)
            self.random_pause(0.5, 2.0)
            comment_area.send_keys(Keys.RETURN)
            self.random_pause(0.5, 2.0)
            logger.info(f'Comment {comment_count} posted: {comment}')
        except Exception as e:
            logger.error(f'Error during comment posting: {e}')
            raise
print('FacebookAICommentBot class defined.')

### Step 1: Initialize Bot and Setup Driver
This will open a Chrome browser window.

In [ ]:
bot = FacebookAICommentBot()
bot.setup_driver()

### Step 2: Navigate to Post URL
Ensure you are logged in if necessary. You can log in manually in the opened browser window.

In [ ]:
bot.driver.get(bot.config['POST_URL'])
print(f'Navigated to: {bot.config["POST_URL"]}')

### Step 3: Generate a Comment via OpenAI

In [ ]:
generated_comment = bot.generate_comment()
print(f'Generated Comment: {generated_comment}')

### Step 4: Test Human-like Typing and Post Comment
This will actually simulate the typing and clicking to post the comment. Watch the browser!

In [ ]:
bot.post_comment(generated_comment, comment_count=1)

### Step 5: Cleanup
Close the browser window after testing.

In [ ]:
bot.driver.quit()
print('Browser closed.')